# Run a review with `run_review`

The smallest way to drive the whole pipeline from Python: build a `ReviewRequest`, call
`agentic_review.agents.pipeline_agent.run_review`, and look at what comes back. This is the same call the
`agentic-review review` command makes.

- Select this project's `.venv` kernel (`uv sync` installs `ipykernel`).
- Put the provider key in `.env` as `AGENTIC_REVIEW_API_KEY` (`GROQ_API_KEY` is used as a fallback).
- `REPO` must be a git repository with uncommitted changes. The default is the demo built by
  `scripts/make_review_demo.py`.

## 1 · Settings

In [13]:
import os
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# ---- change these -------------------------------------------------------------------------------------------------
REPO = PROJECT.parent / "review-demo"  # the repository to review
SCOPE = None                           # None = config/default, or "working-tree" | "staged" | "range"
BASE = None                            # the base ref, only for SCOPE = "range"
MODEL = "openai/gpt-oss-120b"  
BASE_URL = "https://api.groq.com/openai/v1"  # NVIDIA NIM (SRS \u00a72.3, same default as AgentSettings.base_url)

# -------------------------------------------------------------------------------------------------------------------


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                values[key.strip()] = value.strip().strip('"')
    return values


ENV = {**os.environ, **read_env_file(PROJECT / ".env")}
if not ENV.get("AGENTIC_REVIEW_API_KEY") and ENV.get("GROQ_API_KEY"):
    ENV["AGENTIC_REVIEW_API_KEY"] = ENV["GROQ_API_KEY"]

print("repo:   ", REPO, "(exists)" if REPO.is_dir() else "(MISSING)")
print("api key:", "set" if ENV.get("AGENTIC_REVIEW_API_KEY") else "NOT SET")  # the value is never printed

repo:    d:\Projects\DEPI-industry-project\review-demo (exists)
api key: set


## 2 · Run the review

`port_factory` is optional. It is used here only so `MODEL` / `BASE_URL` above can override the repository's
configuration without editing any file; leave both `None` and `run_review` uses its default.

In [14]:
import time

from agentic_review.agents.pipeline_agent import ReviewRequest, Reviewed, run_review
from agentic_review.agents.review_port import build_review_port


def override_model(agent_settings, credential):
    overrides = {k: v for k, v in {"model": MODEL, "base_url": BASE_URL}.items() if v}
    return build_review_port(agent_settings.model_copy(update=overrides), credential)


request = ReviewRequest(cwd=REPO, scope=SCOPE, base=BASE)

started = time.perf_counter()
result = run_review(request, env=ENV, port_factory=override_model if (MODEL or BASE_URL) else None)
print(f"{type(result).__name__} in {time.perf_counter() - started:.1f}s")

Reviewed in 7.9s


## 3 · Read the result

`run_review` returns one of two things:

- `PreflightFailure` — the repository, configuration or credential was not usable; nothing was reviewed.
- `Reviewed` — a `ReviewReport`, even when something degraded (FR-11.2). Always read `degradations`.

In [15]:
from rich.console import Console

from agentic_review.core.output.terminal import render_terminal

if not isinstance(result, Reviewed):
    print("PREFLIGHT FAILED:\n" + result.message)
else:
    report = result.report
    render_terminal(report, Console(width=140, force_jupyter=False, force_terminal=False, no_color=True))
    if result.ignored_settings:
        print("ignored settings:", ", ".join(result.ignored_settings))

Scope: uncommitted changes in the working tree (default) · 7 files
Reviewed (4):
  src/format.js
  src/retry.ts
  src/search.ts
  src/users.ts
Listed, not reviewed (3):
  PLANTED_DEFECTS.md (not TypeScript or JavaScript)
  README.md (not TypeScript or JavaScript)
  src/legacy.ts (deleted)
Excluded (1):
  dist/bundle.js (default:build-output)

No checks were run in this build.
No related context was gathered in this build.

src/format.js:2  nit  maintainability
  The variable `unused` is computed but never used, adding dead code that serves no purpose and may confuse readers.
  evidence: diff hunk 0

src/retry.ts:1-10  blocking  error-handling
  The catch block swallows errors and the function returns undefined after all attempts, making failures indistinguishable from a legitimate undefined result and losing error context.
  evidence: diff hunk 0

Findings: 1 blocking / 0 important / 1 nit
Degradation [CF-6] incomplete-file-coverage: No emit_finding call, accepted or rejected, ever nam

In [16]:
assert isinstance(result, Reviewed), "preflight failed - see the cell above"
report = result.report

print("verdict:   ", report.verdict)
print("completion:", report.run.completion_status)
print("model:     ", report.run.model, "@", report.run.provider)
print("tokens:    ", report.run.tokens_in, "in /", report.run.tokens_out, "out")
print("stages ms: ", report.run.stage_durations_ms.model_dump(by_alias=True, exclude_none=True))

print(f"\n{len(report.findings)} finding(s)")
for f in report.findings:
    print(f"- [{f.severity}/{f.category}] {f.file}:{f.start_line}-{f.end_line}  {f.rationale}")

print(f"\n{len(report.degradations)} degradation(s)" + ("  (a clean run)" if not report.degradations else ""))
for d in report.degradations:
    print(f"- {d.source} {d.code}: {d.detail}")

verdict:    blocking
completion: emitted
model:      openai/gpt-oss-120b @ api.groq.com
tokens:     4634 in / 2660 out
stages ms:  {'preflight': 63, 'changeSet': 265, 'assembly': 0, 'review': 7547, 'output': 0}

2 finding(s)
- [nit/maintainability] src/format.js:2-2  The variable `unused` is computed but never used, adding dead code that serves no purpose and may confuse readers.
- [blocking/error-handling] src/retry.ts:1-10  The catch block swallows errors and the function returns undefined after all attempts, making failures indistinguishable from a legitimate undefined result and losing error context.

1 degradation(s)
- CF-6 incomplete-file-coverage: No emit_finding call, accepted or rejected, ever named: src/search.ts, src/users.ts.


## 4 · The full report as JSON

The same document `agentic-review review --json` prints.

In [17]:
from agentic_review.core.output.json_out import to_json

print(to_json(report))

{
  "schemaVersion": "1.0",
  "run": {
    "startedAt": "2026-09-26T16:35:16Z",
    "durationMs": 7875,
    "stageDurationsMs": {
      "preflight": 63,
      "changeSet": 265,
      "assembly": 0,
      "review": 7547,
      "output": 0
    },
    "requestedDepth": null,
    "achievedDepth": null,
    "toolCallCeiling": 25,
    "investigationToolsEnabled": false,
    "toolCallsTotal": 2,
    "toolCallsEmission": 2,
    "toolCallsInvestigation": 0,
    "repairTurns": 1,
    "completionStatus": "emitted",
    "emitPath": "tool-call",
    "model": "openai/gpt-oss-120b",
    "provider": "api.groq.com",
    "sampling": {
      "temperature": 0.0,
      "seed": null
    },
    "tokensIn": 4634,
    "tokensOut": 2660,
    "budgetLimit": 20000,
    "budgetHit": false,
    "schemaVersion": "1.0",
    "toolVersion": "0.1.0"
  },
  "changeSummary": {
    "scope": {
      "kind": "working-tree",
      "baseRef": null,
      "headRef": null,
      "wasDefaulted": true,
      "description": "uncomm